# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/home/zachh/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==3.15.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

I0000 00:00:1784949304.511789   15829 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784949304.556386   15829 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784949306.276508   15829 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.21.0
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

X = df.drop(columns=["Class"]).values
y = df["Class"].values

In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

I0000 00:00:1784949309.199392   15829 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1754 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20


I0000 00:00:1784949310.156445   15951 service.cc:153] XLA service 0x7c8c54e87870 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784949310.156483   15951 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 Ti Laptop GPU, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1784949310.163334   15951 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1784949310.177748   15951 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1784949310.273023   15951 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


13/13 [==============================] - 1s 17ms/step - loss: 0.9933 - accuracy: 0.4646 - val_loss: 0.8093 - val_accuracy: 0.7600
Epoch 2/20
13/13 [==============================] - 0s 7ms/step - loss: 0.6864 - accuracy: 0.8687 - val_loss: 0.5966 - val_accuracy: 0.8400
Epoch 3/20
13/13 [==============================] - 0s 6ms/step - loss: 0.4815 - accuracy: 0.9394 - val_loss: 0.4478 - val_accuracy: 0.8800
Epoch 4/20
13/13 [==============================] - 0s 7ms/step - loss: 0.3431 - accuracy: 0.9697 - val_loss: 0.3423 - val_accuracy: 0.9200
Epoch 5/20
13/13 [==============================] - 0s 7ms/step - loss: 0.2504 - accuracy: 0.9697 - val_loss: 0.2713 - val_accuracy: 0.8800
Epoch 6/20
13/13 [==============================] - 0s 7ms/step - loss: 0.1877 - accuracy: 0.9798 - val_loss: 0.2184 - val_accuracy: 0.9200
Epoch 7/20
13/13 [==============================] - 0s 8ms/step - loss: 0.1454 - accuracy: 0.9798 - val_loss: 0.1800 - val_accuracy: 0.9200
Epoch 8/20
13/13 [============

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

loss, accuracy = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Test Accuracy: {accuracy:.4f}\n")

y_pred_probs = model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

# Classification report
print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=wine.target_names, digits=4))

print("Confusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Test Accuracy: 1.0000

Classification Report:

              precision    recall  f1-score   support

     class_0     1.0000    1.0000    1.0000        19
     class_1     1.0000    1.0000    1.0000        21
     class_2     1.0000    1.0000    1.0000        14

    accuracy                         1.0000        54
   macro avg     1.0000    1.0000    1.0000        54
weighted avg     1.0000    1.0000    1.0000        54

Confusion Matrix:

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# helper from lab3
import os
def save_binary_model(model_content, filename):
    with open(filename, "wb") as f:
        f.write(model_content)
    return os.path.getsize(filename) / 1024.0  # KB

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model_base = converter.convert()

model_base_size_kb = save_binary_model(tflite_model_base, "model_base.tflite")
print(f"Model size: {model_base_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpjo2anp84/assets


INFO:tensorflow:Assets written to: /tmp/tmpjo2anp84/assets
W0000 00:00:1784949313.135301   15829 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1784949313.135334   15829 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1784949313.135535   15829 reader.cc:83] Reading SavedModel from: /tmp/tmpjo2anp84
I0000 00:00:1784949313.136310   15829 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1784949313.136318   15829 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpjo2anp84
I0000 00:00:1784949313.142280   15829 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1784949313.142859   15829 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1784949313.168962   15829 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpjo2anp84
I0000 00:00:1784949313.174606   15829 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Too

Model size: 14.08 KB


## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        
    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
        
    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    tflite_model = converter.convert()
    size_kb = save_binary_model(tflite_model, filename)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_scale, input_zero_point = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]

    y_pred = []
    for i in range(len(X_test)):
        x = X_test[i:i + 1].astype(np.float32)

        if input_details["dtype"] == np.int8:
            x = np.round(x / input_scale + input_zero_point).astype(np.int8)
        elif input_details["dtype"] == np.uint8:
            x = np.round(x / input_scale + input_zero_point).astype(np.uint8)
        else:
            x = x.astype(input_details["dtype"])

        interpreter.set_tensor(input_details["index"], x)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details["index"])

        if output_details["dtype"] in (np.int8, np.uint8):
            output = (output.astype(np.float32) - output_zero_point) * output_scale

        y_pred.append(np.argmax(output, axis=1)[0])

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)


    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {size_kb:.2f} KB")

    print(f"{quant_type.upper()} Test Accuracy: {np.mean(y_pred == y_true):.4f}\n")

    print("Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=wine.target_names, digits=4))

    print("Confusion Matrix:\n")
    print(confusion_matrix(y_true, y_pred))

    return y_pred, size_kb

In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

int8_preds, int8_size_kb = quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8', 'model_int8.tflite')
float16_preds, float16_size_kb = quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')
dynamic_preds, dynamic_size_kb = quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')

INFO:tensorflow:Assets written to: /tmp/tmpw4yas144/assets


INFO:tensorflow:Assets written to: /tmp/tmpw4yas144/assets
/home/zachh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1784949313.579649   15829 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1784949313.579685   15829 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1784949313.579806   15829 reader.cc:83] Reading SavedModel from: /tmp/tmpw4yas144
I0000 00:00:1784949313.580575   15829 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1784949313.580586   15829 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpw4yas144
I0000 00:00:1784949313.585533   15829 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1784949313.610052   15829 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpw4yas144
I0000 0


INT8 TFLite model size: 8.00 KB
INT8 Test Accuracy: 1.0000

Classification Report:

              precision    recall  f1-score   support

     class_0     1.0000    1.0000    1.0000        19
     class_1     1.0000    1.0000    1.0000        21
     class_2     1.0000    1.0000    1.0000        14

    accuracy                         1.0000        54
   macro avg     1.0000    1.0000    1.0000        54
weighted avg     1.0000    1.0000    1.0000        54

Confusion Matrix:

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /tmp/tmpllmduceg/assets


INFO:tensorflow:Assets written to: /tmp/tmpllmduceg/assets
W0000 00:00:1784949314.112379   15829 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1784949314.112418   15829 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1784949314.112538   15829 reader.cc:83] Reading SavedModel from: /tmp/tmpllmduceg
I0000 00:00:1784949314.113335   15829 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1784949314.113344   15829 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpllmduceg
I0000 00:00:1784949314.117988   15829 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1784949314.142049   15829 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpllmduceg
I0000 00:00:1784949314.148184   15829 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 35654 microseconds.
/home/zachh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/int


FLOAT16 TFLite model size: 8.95 KB
FLOAT16 Test Accuracy: 1.0000

Classification Report:

              precision    recall  f1-score   support

     class_0     1.0000    1.0000    1.0000        19
     class_1     1.0000    1.0000    1.0000        21
     class_2     1.0000    1.0000    1.0000        14

    accuracy                         1.0000        54
   macro avg     1.0000    1.0000    1.0000        54
weighted avg     1.0000    1.0000    1.0000        54

Confusion Matrix:

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /tmp/tmp8mvnajta/assets


INFO:tensorflow:Assets written to: /tmp/tmp8mvnajta/assets
W0000 00:00:1784949314.624938   15829 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1784949314.624976   15829 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1784949314.625095   15829 reader.cc:83] Reading SavedModel from: /tmp/tmp8mvnajta
I0000 00:00:1784949314.625578   15829 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1784949314.625585   15829 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp8mvnajta
I0000 00:00:1784949314.628878   15829 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1784949314.652999   15829 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp8mvnajta
I0000 00:00:1784949314.659513   15829 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 34426 microseconds.
/home/zachh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/int


DYNAMIC TFLite model size: 8.55 KB
DYNAMIC Test Accuracy: 1.0000

Classification Report:

              precision    recall  f1-score   support

     class_0     1.0000    1.0000    1.0000        19
     class_1     1.0000    1.0000    1.0000        21
     class_2     1.0000    1.0000    1.0000        14

    accuracy                         1.0000        54
   macro avg     1.0000    1.0000    1.0000        54
weighted avg     1.0000    1.0000    1.0000        54

Confusion Matrix:

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruning_epochs = 10
pruning_batch_size = 8

steps_per_epoch = int(np.ceil((len(X_train_scaled) * 0.8) / pruning_batch_size))
end_step = steps_per_epoch * pruning_epochs

pruning_params = {
    "pruning_schedule": tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.5,
        final_sparsity=0.7,
        begin_step=0,
        end_step=end_step
    )
}

print("Pruning end_step:", end_step)

Pruning end_step: 130


In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

pruned_model = Sequential([
    prune_low_magnitude(Dense(64, activation='relu', input_shape=(num_features,)), **pruning_params),
    prune_low_magnitude(Dense(32, activation='relu'), **pruning_params),
    prune_low_magnitude(Dense(num_classes, activation='softmax'), **pruning_params),
])

pruned_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 3 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 4 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 5 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

pruning_callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

pruning_history = pruned_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=pruning_epochs,
    batch_size=pruning_batch_size,
    validation_split=0.2,
    callbacks=pruning_callbacks,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 3s 18ms/step - loss: 0.9231 - accuracy: 0.6364 - val_loss: 0.7132 - val_accuracy: 0.9200
Epoch 2/10
13/13 [==============================] - 0s 8ms/step - loss: 0.6649 - accuracy: 0.8889 - val_loss: 0.5204 - val_accuracy: 0.9600
Epoch 3/10
13/13 [==============================] - 0s 8ms/step - loss: 0.4687 - accuracy: 0.9798 - val_loss: 0.3686 - val_accuracy: 1.0000
Epoch 4/10
13/13 [==============================] - 0s 9ms/step - loss: 0.3232 - accuracy: 0.9899 - val_loss: 0.2500 - val_accuracy: 1.0000
Epoch 5/10
13/13 [==============================] - 0s 10ms/step - loss: 0.2233 - accuracy: 0.9899 - val_loss: 0.1746 - val_accuracy: 1.0000
Epoch 6/10
13/13 [==============================] - 0s 8ms/step - loss: 0.1618 - accuracy: 0.9899 - val_loss: 0.1334 - val_accuracy: 1.0000
Epoch 7/10
13/13 [==============================] - 0s 9ms/step - loss: 0.1210 - accuracy: 0.9899 - val_loss: 0.1057 - val_accuracy: 1.0000
Epoch 8/10
13/13 [

In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
converter.optimizations = [tf.lite.Optimize.EXPERIMENTAL_SPARSITY]
tflite_pruned_model = converter.convert()

model_pruned_size_kb= save_binary_model(tflite_pruned_model, "model_pruned.tflite")

print(f"Pruned TFLite model size: {model_pruned_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmp0v1a3wsr/assets


INFO:tensorflow:Assets written to: /tmp/tmp0v1a3wsr/assets


Pruned TFLite model size: 8.08 KB


W0000 00:00:1784949321.783314   15829 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1784949321.783346   15829 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1784949321.783456   15829 reader.cc:83] Reading SavedModel from: /tmp/tmp0v1a3wsr
I0000 00:00:1784949321.783799   15829 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1784949321.783807   15829 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp0v1a3wsr
I0000 00:00:1784949321.785849   15829 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1784949321.795696   15829 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp0v1a3wsr
I0000 00:00:1784949321.798996   15829 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 15546 microseconds.
W0000 00:00:1784949321.841784   15829 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.


In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

probs = stripped_pruned_model.predict(X_test_scaled, verbose=0)
preds = np.argmax(probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print(f"Pruned Model Test Accuracy: {np.mean(preds == y_true):.4f}\n")

print("Classification Report:\n")
print(classification_report(y_true, preds, target_names=wine.target_names, digits=4))

print("Confusion Matrix:\n")
print(confusion_matrix(y_true, preds))

Pruned Model Test Accuracy: 1.0000

Classification Report:

              precision    recall  f1-score   support

     class_0     1.0000    1.0000    1.0000        19
     class_1     1.0000    1.0000    1.0000        21
     class_2     1.0000    1.0000    1.0000        14

    accuracy                         1.0000        54
   macro avg     1.0000    1.0000    1.0000        54
weighted avg     1.0000    1.0000    1.0000        54

Confusion Matrix:

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 32)                448       
                                                                 
 dense_7 (Dense)             (None, 16)                528       
                                                                 
 dense_8 (Dense)             (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = model.predict(X_train_scaled, verbose=0)
print("Teacher soft labels shape:", teacher_preds_soft.shape)

Teacher soft labels shape: (124, 3)


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)

alpha = 0.5

def distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return alpha * hard_loss + (1.0 - alpha) * soft_loss

In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer='adam',
    loss=distillation_loss
)

distillation_history = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 1s 18ms/step - loss: 1.2284 - val_loss: 1.0285
Epoch 2/10
13/13 [==============================] - 0s 9ms/step - loss: 0.9590 - val_loss: 0.8166
Epoch 3/10
13/13 [==============================] - 0s 10ms/step - loss: 0.7749 - val_loss: 0.6682
Epoch 4/10
13/13 [==============================] - 0s 10ms/step - loss: 0.6362 - val_loss: 0.5620
Epoch 5/10
13/13 [==============================] - 0s 9ms/step - loss: 0.5264 - val_loss: 0.4828
Epoch 6/10
13/13 [==============================] - 0s 9ms/step - loss: 0.4384 - val_loss: 0.4190
Epoch 7/10
13/13 [==============================] - 0s 8ms/step - loss: 0.3697 - val_loss: 0.3619
Epoch 8/10
13/13 [==============================] - 0s 9ms/step - loss: 0.3119 - val_loss: 0.3177
Epoch 9/10
13/13 [==============================] - 0s 9ms/step - loss: 0.2671 - val_loss: 0.2799
Epoch 10/10
13/13 [==============================] - 0s 8ms/step - loss: 0.2284 - val_loss: 0.2520


In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd_model = converter.convert()

model_kd_size_kb = save_binary_model(tflite_kd_model, "model_kd.tflite")

print(f"Knowledge-distilled TFLite model size: {model_kd_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpbk9_70sj/assets


INFO:tensorflow:Assets written to: /tmp/tmpbk9_70sj/assets
W0000 00:00:1784949324.736202   15829 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1784949324.736244   15829 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1784949324.736363   15829 reader.cc:83] Reading SavedModel from: /tmp/tmpbk9_70sj
I0000 00:00:1784949324.736935   15829 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1784949324.736943   15829 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpbk9_70sj
I0000 00:00:1784949324.740650   15829 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1784949324.763657   15829 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpbk9_70sj
I0000 00:00:1784949324.771225   15829 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 34874 microseconds.


Knowledge-distilled TFLite model size: 6.12 KB


In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

probs = student_model.predict(X_test_scaled, verbose=0)
preds = np.argmax(probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print(f"Distilled Student Test Accuracy: {np.mean(preds == y_true):.4f}\n")

print("Classification Report:\n")
print(classification_report(y_true, preds, target_names=wine.target_names, digits=4))

print("Confusion Matrix:\n")
print(confusion_matrix(y_true, preds))

Distilled Student Test Accuracy: 0.9815

Classification Report:

              precision    recall  f1-score   support

     class_0     1.0000    1.0000    1.0000        19
     class_1     1.0000    0.9524    0.9756        21
     class_2     0.9333    1.0000    0.9655        14

    accuracy                         0.9815        54
   macro avg     0.9778    0.9841    0.9804        54
weighted avg     0.9827    0.9815    0.9816        54

Confusion Matrix:

[[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [27]:
student_for_pruning = tf.keras.models.clone_model(student_model)
student_for_pruning.set_weights(student_model.get_weights())

final_pruned_student = prune_low_magnitude(student_for_pruning, **pruning_params)

final_pruned_student.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

final_pruned_student.fit(
    X_train_scaled,
    y_train_cat,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=1
)

stripped_final_model = tfmot.sparsity.keras.strip_pruning(final_pruned_student)

final_preds, final_size_kb = quantize_and_evaluate(
    stripped_final_model, X_test_scaled, y_test_cat, 'int8', 'model_final_pruned_int8.tflite'
)

print("\nSummary of TFLite model sizes (KB):")
print(f"  Base model:                {model_base_size_kb:.2f}")
print(f"  Pruned model:              {model_pruned_size_kb:.2f}")
print(f"  Distilled student (KD):    {model_kd_size_kb:.2f}")
print(f"  Distilled + pruned + int8: {final_size_kb:.2f}")


Epoch 1/10
13/13 [==============================] - 2s 22ms/step - loss: 0.1802 - accuracy: 1.0000 - val_loss: 0.1898 - val_accuracy: 0.9600
Epoch 2/10
13/13 [==============================] - 0s 10ms/step - loss: 0.1415 - accuracy: 0.9899 - val_loss: 0.1644 - val_accuracy: 0.9600
Epoch 3/10
13/13 [==============================] - 0s 10ms/step - loss: 0.1123 - accuracy: 0.9899 - val_loss: 0.1415 - val_accuracy: 0.9600
Epoch 4/10
13/13 [==============================] - 0s 11ms/step - loss: 0.0918 - accuracy: 1.0000 - val_loss: 0.1205 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 14ms/step - loss: 0.0745 - accuracy: 1.0000 - val_loss: 0.1077 - val_accuracy: 1.0000
Epoch 6/10
13/13 [==============================] - 0s 13ms/step - loss: 0.0624 - accuracy: 1.0000 - val_loss: 0.0968 - val_accuracy: 1.0000
Epoch 7/10
13/13 [==============================] - 0s 10ms/step - loss: 0.0527 - accuracy: 1.0000 - val_loss: 0.0873 - val_accuracy: 1.0000
Epoch 8/10
13

INFO:tensorflow:Assets written to: /tmp/tmpyppxm6zk/assets



INT8 TFLite model size: 4.77 KB
INT8 Test Accuracy: 0.9815

Classification Report:

              precision    recall  f1-score   support

     class_0     1.0000    1.0000    1.0000        19
     class_1     1.0000    0.9524    0.9756        21
     class_2     0.9333    1.0000    0.9655        14

    accuracy                         0.9815        54
   macro avg     0.9778    0.9841    0.9804        54
weighted avg     0.9827    0.9815    0.9816        54

Confusion Matrix:

[[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]

Summary of TFLite model sizes (KB):
  Base model:                14.08
  Pruned model:              8.08
  Distilled student (KD):    6.12
  Distilled + pruned + int8: 4.77


/home/zachh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1784949328.258365   15829 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1784949328.258401   15829 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1784949328.258519   15829 reader.cc:83] Reading SavedModel from: /tmp/tmpyppxm6zk
I0000 00:00:1784949328.258951   15829 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1784949328.258957   15829 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpyppxm6zk
I0000 00:00:1784949328.260952   15829 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1784949328.270416   15829 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpyppxm6zk
I0000 00:00:1784949328.273925   15829 loader.cc:471] SavedModel lo

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
